# Python Toolkit — seven tools that make Python click

Companion notebook for **[python_toolkit.html](python_toolkit.html)**.

You already know functions, loops and lists. These are the next seven tools.
Each one solves a problem you will meet soon.

| # | Tool | Solves |
|---|------|--------|
| 1 | `class` | keeping data and behaviour together |
| 2 | `yield` (generators) | data too big for memory |
| 3 | `@decorator` | adding behaviour without editing a function |
| 4 | type hints | saying what goes in and what comes out |
| 5 | `pathlib` | file paths that work on every computer |
| 6 | `json` | saving and loading data |
| 7 | `requests` | fetching web pages and APIs |

**How to use this notebook**

- Run each cell with **Shift + Enter**.
- Run the cells **in order** — later ones use names defined earlier.
- Every section ends with an exercise. Try it before looking at the answers in section 10.
- Change the code. Break it on purpose. Read the error. That is the fastest way to learn.

Only section 7 needs the internet, and one install:

```
pip install requests
```

---
## 1. Classes — data and behaviour in one place

With plain variables, two dogs need four variables and nothing links a name to its age:

```python
dog1_name = "Somchai"
dog1_age = 3
dog2_name = "Nong Mee"
dog2_age = 5
```

A **class** is a template. It says what data one dog holds, and what one dog can do.

In [ ]:
class Dog:
    """One dog at the shelter."""

    def __init__(self, name, age):
        self.name = name
        self.age = age

    def bark(self):
        return f"{self.name} says woof!"

    def human_years(self):
        return self.age * 7


# Each dog made from the template is an "object"
somchai = Dog("Somchai", 3)
nong_mee = Dog("Nong Mee", 5)

print(somchai.bark())
print(nong_mee.bark())
print(nong_mee.name, "is", nong_mee.human_years(), "in human years")

Reading it:

- `class Dog:` starts the template. Class names use CapitalCase.
- `__init__` runs once when you make a new dog. It is the *constructor*.
- `self` means "this particular dog". Every method takes it first.
- `self.name = name` stores the name **on the object**, so it stays there.
- You never pass `self` yourself — `somchai.bark()` passes it for you.

Each object keeps its own data. Proving it:

In [ ]:
print(somchai.age, nong_mee.age)

somchai.age = 4          # only this dog changes
print(somchai.age, nong_mee.age)

print(type(somchai))     # <class '__main__.Dog'>

`__str__` controls what `print(dog)` shows. Without it you get an ugly memory address.

In [ ]:
print(somchai)           # ugly


class Dog:
    """One dog at the shelter, now printable."""

    def __init__(self, name, age):
        self.name = name
        self.age = age

    def bark(self):
        return f"{self.name} says woof!"

    def human_years(self):
        return self.age * 7

    def __str__(self):
        return f"Dog({self.name}, {self.age} years old)"


somchai = Dog("Somchai", 3)
print(somchai)           # much better

### Exercise 1

Write a class `Student` with:

- `__init__` taking a `name` and a list of `scores`
- a method `average()` returning the mean score
- a method `passed()` returning `True` if the average is 60 or more
- a `__str__` so `print(s)` shows something readable

Then make a student with scores `[82, 91, 77]` and print the average.

In [ ]:
# Your code here

---
## 2. Generators — `yield` for data that will not fit

`return` hands back everything at once. For a 20 GB file that means running out of memory.
`yield` hands back **one item at a time**.

Watch the difference. First, a normal function that returns a list:

In [ ]:
def squares_list(n):
    result = []
    for i in range(n):
        result.append(i * i)
    return result          # the WHOLE list, all at once


def squares_gen(n):
    for i in range(n):
        yield i * i        # give ONE value, then pause


print(squares_list(5))
print(squares_gen(5))      # not a list — a generator object!

Calling a generator function does **not** run it. It returns a promise to produce values later.
The code only runs when something asks for a value.

In [ ]:
gen = squares_gen(5)

print(next(gen))    # 0  — runs until the first yield, then pauses
print(next(gen))    # 1  — wakes up exactly where it stopped
print(next(gen))    # 4

print("the rest:", list(gen))   # 9, 16 — only what is left

Now the memory difference, measured. `sys.getsizeof` reports how many bytes an object holds.

In [ ]:
import sys

big_list = [i * i for i in range(1_000_000)]
big_gen = (i * i for i in range(1_000_000))   # round brackets = generator expression

print("list:     ", sys.getsizeof(big_list), "bytes")
print("generator:", sys.getsizeof(big_gen), "bytes")

The list grows with the data. The generator does not — it only remembers where it paused.

Because values are made on demand, a generator can be **endless**. A list cannot.

In [ ]:
def count_from(start):
    n = start
    while True:        # never ends
        yield n
        n += 1


for n in count_from(10):
    print(n)
    if n >= 13:
        break          # you must break out yourself

**Three rules to remember:**

1. Any function containing `yield` is a generator. There is no other keyword.
2. A generator is used **once**. Loop over it twice and the second loop gets nothing.
3. You cannot use `len()` on it or index it with `[3]`. It does not know its own size.

Rule 2 catches everybody. See it happen:

In [ ]:
gen = squares_gen(5)

print("first loop: ", list(gen))
print("second loop:", list(gen))   # empty! the generator is used up

### Exercise 2

Write a generator `even_numbers(limit)` that yields even numbers from 0 up to (but not
including) `limit`.

Then use it to print the even numbers below 10.

**Bonus:** write `first_n(gen, n)` that takes any generator and yields only its first `n`
values — so `list(first_n(count_from(100), 3))` gives `[100, 101, 102]`.

In [ ]:
# Your code here

---
## 3. Decorators — `@something` wraps a function

You have five functions and want to time all of them. Copying two lines into each is the
wrong answer. A **decorator** writes those lines once and wraps them around any function.

A decorator is a function that takes a function and returns a new function. That sentence
is the whole idea.

In [ ]:
import time


def timed(fn):
    """Print how long the wrapped function takes."""
    def wrapper(*args, **kwargs):
        start = time.time()
        result = fn(*args, **kwargs)
        print(f"[{fn.__name__} took {time.time() - start:.2f}s]")
        return result          # forget this and your function returns None
    return wrapper             # no brackets — return the function itself


@timed
def slow_add(a, b):
    time.sleep(1)
    return a + b


print(slow_add(2, 3))

The `@` is only shorthand. These two are exactly the same:

```python
@timed
def slow_add(a, b): ...

# ...means:

def slow_add(a, b): ...
slow_add = timed(slow_add)
```

Proving it, with no `@` anywhere:

In [ ]:
def plain_add(a, b):
    time.sleep(0.5)
    return a + b


plain_add = timed(plain_add)    # exactly what @timed does
print(plain_add(10, 20))

`*args, **kwargs` means "accept any inputs and pass them straight through". That is what
lets one decorator wrap functions with completely different signatures.

In [ ]:
@timed
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"


print(greet("Ploy"))
print(greet("Ploy", greeting="Sawasdee"))

### One line worth adding

Without help, a decorated function forgets its own name and docstring:

In [ ]:
print(slow_add.__name__)   # "wrapper" — wrong!
print(slow_add.__doc__)    # the wrapper's docstring, not slow_add's

import functools


def timed(fn):
    """Print how long the wrapped function takes."""
    @functools.wraps(fn)           # <-- the fix
    def wrapper(*args, **kwargs):
        start = time.time()
        result = fn(*args, **kwargs)
        print(f"[{fn.__name__} took {time.time() - start:.2f}s]")
        return result
    return wrapper


@timed
def slow_add(a, b):
    """Add two numbers, slowly."""
    time.sleep(0.2)
    return a + b


slow_add(1, 2)
print(slow_add.__name__)   # "slow_add" — correct
print(slow_add.__doc__)    # its own docstring

You will use decorators from libraries long before you write your own. Here is a real one
from the standard library — `@lru_cache` remembers past results:

In [ ]:
from functools import lru_cache


@lru_cache
def fib(n):
    """The nth Fibonacci number, the slow recursive way."""
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)


start = time.time()
print(fib(35))
print(f"{time.time() - start:.4f}s")   # instant — try deleting @lru_cache and rerunning

### Exercise 3

Write a decorator `shout` that turns whatever the function returns into upper case,
and adds `"!!!"` at the end.

```python
@shout
def greet(name):
    return f"hello {name}"

greet("ploy")     # should give "HELLO PLOY!!!"
```

**Bonus:** write `count_calls`, which prints how many times the function has been called.

In [ ]:
# Your code here

---
## 4. Type hints — say what goes in and what comes out

Look at `def process(data):` — what is `data`? A list? A file? A number? You have to read
the whole body to find out. Type hints answer it in the first line.

In [ ]:
def greet(name: str) -> str:
    """Return a friendly greeting."""
    return f"Hello, {name}!"


print(greet("Ploy"))

# Python does NOT enforce hints. This still runs:
print(greet(42))

Hints are notes for humans and tools, not rules for the interpreter. They are still worth
writing, because your editor, `mypy`, and AI coding tools all read them.

The types you will actually use:

| Hint | Means |
|------|-------|
| `str`, `int`, `float`, `bool` | text, whole number, decimal, True/False |
| `list[str]` | a list of strings |
| `dict[str, int]` | dictionary with string keys, number values |
| `str \| None` | a string, or nothing — very common for "might fail" |
| `Path` | a `pathlib` path |
| `-> None` | returns nothing (only prints or saves) |

In [ ]:
from pathlib import Path


def word_count(text: str) -> dict[str, int]:
    """Count how many times each word appears."""
    counts: dict[str, int] = {}
    for word in text.lower().split():
        counts[word] = counts.get(word, 0) + 1
    return counts


def find_config(folder: Path) -> Path | None:
    """Return the config file if it exists, otherwise None."""
    candidate = folder / "config.json"
    return candidate if candidate.exists() else None


def announce(message: str) -> None:
    """Print a message. Returns nothing."""
    print(f">>> {message}")


print(word_count("the cat sat on the mat the end"))
print(find_config(Path(".")))     # None, unless you happen to have one
announce("hints are just notes")

Hints are stored on the function, which is exactly how tools read them:

In [ ]:
print(greet.__annotations__)
print(word_count.__annotations__)

### Exercise 4

Add type hints to these three functions. Do not change what they do.

```python
def total(prices):
    return sum(prices)

def initials(first, last):
    return first[0] + last[0]

def lookup(names, key):
    # returns a name, or None if the key is missing
    return names.get(key)
```

In [ ]:
# Your code here

---
## 5. pathlib — file paths that work everywhere

Old code joins paths with strings and `os.path`. `pathlib` gives you a real **Path object**,
and reuses the `/` operator to join paths.

In [ ]:
from pathlib import Path

base = Path("toolkit_demo")
notes = base / "lessons" / "notes.txt"

print(notes)              # Windows and Mac each get the right separator
print(notes.name)         # notes.txt
print(notes.stem)         # notes
print(notes.suffix)       # .txt
print(notes.parent)       # toolkit_demo/lessons
print(notes.exists())     # False — nothing created yet

Nothing has touched the disk yet. A `Path` is just a description of a location.

Now create the folder and write a file. `parents=True` makes any missing parent folders;
`exist_ok=True` means "do not complain if it is already there".

In [ ]:
notes.parent.mkdir(parents=True, exist_ok=True)

notes.write_text("Sawasdee from pathlib!\nLine two.\n", encoding="utf-8")

print(notes.exists())
print(notes.read_text(encoding="utf-8"))

**Always pass `encoding="utf-8"`.** On Windows, Python may otherwise use a local encoding,
and Thai or Chinese text comes back broken. One keyword prevents a whole class of bug.

In [ ]:
thai = base / "lessons" / "thai.txt"
thai.write_text("ครูสอนภาษาอังกฤษ\n老师\n", encoding="utf-8")
print(thai.read_text(encoding="utf-8"))

`glob` finds files by pattern. `rglob` does the same through every sub-folder.
Both return generators — section 2 again.

In [ ]:
# make a few more files to search
for i in range(3):
    (base / "lessons" / f"day{i}.txt").write_text(f"day {i}\n", encoding="utf-8")

print("glob is lazy:", base.rglob("*.txt"))
print()

for txt in sorted(base.rglob("*.txt")):
    size = len(txt.read_text(encoding="utf-8"))
    print(f"{txt.stem:10} {size:3} characters   ({txt})")

### Handy paths you did not have to type

In [ ]:
print("current folder:", Path.cwd())
print("your home:     ", Path.home())
print("absolute:      ", notes.resolve())
print("is it a file?  ", notes.is_file())
print("is it a folder?", notes.parent.is_dir())

### Exercise 5

Using `pathlib` only (no `os.path`):

1. Make a folder `toolkit_demo/reports`.
2. Write three files into it: `jan.txt`, `feb.txt`, `mar.txt`, each containing one line of text.
3. Loop over them and print `stem` and the number of characters in each.
4. Print the **total** number of characters across all three.

In [ ]:
# Your code here

---
## 6. json — save and load data in two lines

Your program has a dictionary. You want it to still exist tomorrow. JSON is the standard
text format for that. Every language reads it, and every web API speaks it.

In [ ]:
import json
from pathlib import Path

student = {
    "name": "Ploy",
    "level": "B1",
    "scores": [82, 91, 77],
    "active": True,
    "teacher": None,
}

out = Path("toolkit_demo") / "student.json"

# Save it
out.write_text(json.dumps(student, indent=2), encoding="utf-8")

# Load it back
loaded = json.loads(out.read_text(encoding="utf-8"))

print(loaded)
print(loaded["scores"][1])
print(type(loaded))

Look at the file Python actually wrote. Note `true` and `null` in lower case — that is JSON,
not Python.

In [ ]:
print(out.read_text(encoding="utf-8"))

### Four names, easy to mix up

| Function | Direction | Remember it as |
|---|---|---|
| `json.dumps(obj)` | Python → text | dump-**s** = dump to **s**tring |
| `json.loads(text)` | text → Python | load-**s** = load from **s**tring |
| `json.dump(obj, f)` | Python → open file | no s = works on a file |
| `json.load(f)` | open file → Python | no s = works on a file |

With `pathlib` you mostly need only the two `s` versions.

### The argument that matters for Thai and Chinese

In [ ]:
lesson = {"teacher": "ครู", "subject": "老师"}

print("default:          ", json.dumps(lesson))
print("ensure_ascii=False:", json.dumps(lesson, ensure_ascii=False))

Both load back correctly, but only one is readable by a human opening the file.

### What JSON cannot hold

Dates, sets, and your own class objects are not JSON types. Convert them first.

In [ ]:
from datetime import date

bad = {"today": date.today(), "tags": {"a", "b"}}

try:
    json.dumps(bad)
except TypeError as e:
    print("TypeError:", e)

# Convert first:
good = {"today": str(date.today()), "tags": list({"a", "b"})}
print(json.dumps(good))

### Exercise 6

1. Build a list of three dictionaries, each with `name` and `score`.
2. Save it to `toolkit_demo/class.json` with `indent=2` and `ensure_ascii=False`.
3. Load it back into a new variable.
4. Print the name of the student with the highest score.

**Hint for step 4:** `max(students, key=lambda s: s["score"])`

In [ ]:
# Your code here

---
## 7. requests — fetch web pages and APIs

The first six tools ship with Python. This one you install:

```
pip install requests
```

**This section needs an internet connection.** If a cell fails with a connection error,
that is the network, not your code.

In [ ]:
import requests

r = requests.get("https://api.github.com/repos/python/cpython", timeout=10)
r.raise_for_status()
data = r.json()

print(data["name"], "has", data["stargazers_count"], "stars")
print("language:", data["language"])

Four lines, and every one earns its place:

- `requests.get(url)` asks a server for something and waits.
- `timeout=10` gives up after 10 seconds. **Always set it** or your program can hang forever.
- `raise_for_status()` raises an error on 404 or 500. Without it, a failed request looks successful.
- `r.json()` turns the reply into Python dicts and lists — it runs `json.loads` for you.

What comes back:

In [ ]:
print("status:  ", r.status_code)   # 200 is fine
print("ok:      ", r.ok)
print("type:    ", r.headers["content-type"])
print("text len:", len(r.text), "characters of raw JSON")

See what `raise_for_status()` protects you from:

In [ ]:
bad = requests.get("https://api.github.com/repos/python/this-does-not-exist", timeout=10)

print("status:", bad.status_code)     # 404 — but no crash yet!
print("ok:    ", bad.ok)

try:
    bad.raise_for_status()
except requests.HTTPError as e:
    print("caught:", e)

### Sending data with the request

`params=` builds the `?key=value` part safely. Never glue it on with `+`.

In [ ]:
r = requests.get("https://dummyjson.com/quotes", params={"limit": 3}, timeout=10)
r.raise_for_status()

print("asked for:", r.url)
print()

for q in r.json()["quotes"]:
    print(f'"{q["quote"]}"')
    print(f'   — {q["author"]}')

`json=` sends a Python dictionary as a JSON body, and sets the header for you.
`headers=` is where API keys go.

`httpbin.org` echoes back whatever you send, which makes it perfect for seeing what left
your computer. It is a free shared server, so it is sometimes busy — `RequestException` is
the parent of every error `requests` can raise, so catching it catches them all.

In [ ]:
try:
    r = requests.post(
        "https://httpbin.org/post",
        json={"student": "Ploy", "score": 91},
        headers={"X-Demo-Header": "hello"},
        timeout=15,
    )
    r.raise_for_status()
    echo = r.json()

    print("server saw this body:  ", echo["json"])
    print("server saw this header:", echo["headers"].get("X-Demo-Header"))

except requests.RequestException as e:
    print("httpbin.org is busy. That is the server, not your code — try again in a moment.")
    print("error:", e)

**Never put an API key in your code.** Read it from the environment instead:

```python
import os
token = os.getenv("MY_API_KEY")
```

A key pasted into a file will end up on GitHub one day.

### requests + pathlib: saving a download

In [ ]:
from pathlib import Path

r = requests.get("https://www.python.org/static/img/python-logo.png", timeout=30)
r.raise_for_status()

logo = Path("toolkit_demo") / "logo.png"
logo.write_bytes(r.content)          # .content is raw bytes, not text

print("saved", logo, "-", logo.stat().st_size, "bytes")

### Exercise 7

The GitHub API can list a user's public repositories:

`https://api.github.com/users/USERNAME/repos`

1. Fetch the repos for the user `python`.
2. Use `params={"per_page": 5}` so you only get five.
3. Set a `timeout`, and call `raise_for_status()`.
4. Print each repo's `name` and `stargazers_count`.
5. Save the whole reply to `toolkit_demo/repos.json` — that is section 6 again.

In [ ]:
# Your code here

---
## 8. All seven in one small program

This script fetches quotes from a public API, saves them, and prints them. Every one of the
seven tools appears — the comments mark them.

Run it twice. The **first** run fetches from the internet. The **second** reads the file and
is instant. That difference is what a cache is.

In [ ]:
import json
import time
from pathlib import Path
from typing import Iterator

import requests

CACHE_DIR = Path("toolkit_demo")


def timed(fn):                                          # 3. decorator
    """Print how long the wrapped function takes."""
    def wrapper(*args, **kwargs):
        start = time.time()
        result = fn(*args, **kwargs)
        print(f"[{fn.__name__} took {time.time() - start:.2f}s]")
        return result
    return wrapper


class Quote:                                            # 1. class
    """One quote and the person who said it."""

    def __init__(self, text: str, author: str):         # 4. type hints
        self.text = text
        self.author = author

    def short(self, limit: int = 50) -> str:
        """The quote, cut to `limit` characters."""
        if len(self.text) <= limit:
            return self.text
        return self.text[:limit - 3] + "..."


@timed
def fetch_quotes(count: int) -> list[dict]:             # 7. requests
    """Ask the API for `count` quotes."""
    r = requests.get(
        "https://dummyjson.com/quotes",
        params={"limit": count},
        timeout=10,
    )
    r.raise_for_status()
    return r.json()["quotes"]


def read_quotes(path: Path) -> Iterator[Quote]:         # 2. generator
    """Read the saved file, one Quote at a time."""
    data = json.loads(path.read_text(encoding="utf-8"))  # 6. json
    for item in data:
        yield Quote(item["quote"], item["author"])


def main() -> None:
    CACHE_DIR.mkdir(exist_ok=True)                      # 5. pathlib
    cache_file = CACHE_DIR / "quotes.json"

    if not cache_file.exists():
        print("No cache. Fetching...")
        quotes = fetch_quotes(5)
        cache_file.write_text(
            json.dumps(quotes, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
    else:
        print(f"Reading cache: {cache_file}")

    for quote in read_quotes(cache_file):
        print(f"  {quote.short()} - {quote.author}")


main()

Run the cell above a second time and watch the first line change.

### Exercise 8

Point the same program at a different API. Change **only**:

- the URL inside `fetch_quotes`
- the `params`
- the keys read in `read_quotes`

Try the GitHub repos endpoint from exercise 7, and make `Quote` into a `Repo` class with
`name` and `stars`. Almost nothing else needs to change — that is the point of writing it
this way.

In [ ]:
# Your code here

### Clean up

When you are finished, this removes the folder the notebook created.

In [ ]:
import shutil
from pathlib import Path

demo = Path("toolkit_demo")
if demo.exists():
    shutil.rmtree(demo)
    print("removed", demo.resolve())
else:
    print("nothing to remove")

---
## 9. One-page cheat sheet

```python
# 1. class — data + behaviour together
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age
    def bark(self):
        return f"{self.name} says woof!"

# 2. generator — one item at a time, any size
def read_lines(path):
    with open(path) as f:
        for line in f:
            yield line.strip()

# 3. decorator — wrap a function
def timed(fn):
    def wrapper(*args, **kwargs):
        ...
        return fn(*args, **kwargs)
    return wrapper

@timed
def slow(): ...

# 4. type hints — what goes in, what comes out
def greet(name: str) -> str: ...
def load(p: Path) -> dict[str, int] | None: ...

# 5. pathlib — paths that work everywhere
p = Path("data") / "notes.txt"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(text, encoding="utf-8")
for f in Path(".").rglob("*.txt"): ...

# 6. json — save and load
p.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
obj = json.loads(p.read_text(encoding="utf-8"))

# 7. requests — fetch the web
r = requests.get(url, params={...}, timeout=10)
r.raise_for_status()
data = r.json()
```

**The five mistakes that cost beginners the most time**

1. No `timeout=` on a request — the program hangs forever.
2. No `raise_for_status()` — a 404 quietly looks like success.
3. No `encoding="utf-8"` — Thai and Chinese text turns to nonsense on Windows.
4. Looping over a generator twice — the second loop silently gets nothing.
5. Forgetting `return result` in a decorator — the function returns `None`.

---
## 10. Answers

Try each exercise before reading these. There is more than one right answer.

### Answer 1 — `Student` class

In [ ]:
class Student:
    """A student and their scores."""

    def __init__(self, name, scores):
        self.name = name
        self.scores = scores

    def average(self):
        if not self.scores:
            return 0
        return sum(self.scores) / len(self.scores)

    def passed(self):
        return self.average() >= 60

    def __str__(self):
        return f"{self.name}: {self.average():.1f} ({'pass' if self.passed() else 'fail'})"


ploy = Student("Ploy", [82, 91, 77])
print(ploy.average())
print(ploy)

### Answer 2 — `even_numbers` and `first_n`

In [ ]:
def even_numbers(limit):
    for n in range(limit):
        if n % 2 == 0:
            yield n


print(list(even_numbers(10)))


# Bonus
def first_n(gen, n):
    for i, value in enumerate(gen):
        if i >= n:
            return          # 'return' ends a generator early
        yield value


def count_from(start):
    n = start
    while True:
        yield n
        n += 1


print(list(first_n(count_from(100), 3)))

### Answer 3 — `shout` and `count_calls`

In [ ]:
import functools


def shout(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        return result.upper() + "!!!"
    return wrapper


@shout
def greet(name):
    return f"hello {name}"


print(greet("ploy"))


# Bonus
def count_calls(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        print(f"{fn.__name__} call #{wrapper.calls}")
        return fn(*args, **kwargs)
    wrapper.calls = 0          # the count lives on the wrapper
    return wrapper


@count_calls
def ping():
    return "pong"


ping()
ping()
ping()

### Answer 4 — type hints

In [ ]:
def total(prices: list[float]) -> float:
    return sum(prices)


def initials(first: str, last: str) -> str:
    return first[0] + last[0]


def lookup(names: dict[str, str], key: str) -> str | None:
    return names.get(key)


print(total([1.5, 2.5]))
print(initials("Ploy", "Suwan"))
print(lookup({"a": "Anan"}, "b"))

### Answer 5 — pathlib

In [ ]:
from pathlib import Path

reports = Path("toolkit_demo") / "reports"
reports.mkdir(parents=True, exist_ok=True)

for month in ["jan", "feb", "mar"]:
    (reports / f"{month}.txt").write_text(f"Report for {month}.\n", encoding="utf-8")

total_chars = 0
for f in sorted(reports.glob("*.txt")):
    text = f.read_text(encoding="utf-8")
    print(f"{f.stem}: {len(text)} characters")
    total_chars += len(text)

print("total:", total_chars)

### Answer 6 — json

In [ ]:
import json
from pathlib import Path

students = [
    {"name": "Ploy", "score": 91},
    {"name": "ครูอาทิตย์", "score": 77},
    {"name": "Nong Mee", "score": 84},
]

path = Path("toolkit_demo") / "class.json"
path.parent.mkdir(exist_ok=True)
path.write_text(
    json.dumps(students, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

loaded = json.loads(path.read_text(encoding="utf-8"))
best = max(loaded, key=lambda s: s["score"])
print("top student:", best["name"], "with", best["score"])

### Answer 7 — requests

In [ ]:
import json
import requests
from pathlib import Path

r = requests.get(
    "https://api.github.com/users/python/repos",
    params={"per_page": 5},
    timeout=10,
)
r.raise_for_status()
repos = r.json()

for repo in repos:
    print(f"{repo['name']:<20} {repo['stargazers_count']:>7} stars")

out = Path("toolkit_demo") / "repos.json"
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(repos, indent=2, ensure_ascii=False), encoding="utf-8")
print("saved", out)

### Answer 8 — the capstone, pointed somewhere else

Only three things changed: the URL, the params, and the keys.

In [ ]:
import json
import time
from pathlib import Path
from typing import Iterator

import requests

CACHE_DIR = Path("toolkit_demo")


def timed(fn):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = fn(*args, **kwargs)
        print(f"[{fn.__name__} took {time.time() - start:.2f}s]")
        return result
    return wrapper


class Repo:                                             # was Quote
    """One GitHub repository."""

    def __init__(self, name: str, stars: int):
        self.name = name
        self.stars = stars

    def line(self) -> str:
        return f"{self.name:<20} {self.stars:>7} stars"


@timed
def fetch_repos(count: int) -> list[dict]:
    r = requests.get(
        "https://api.github.com/users/python/repos",   # changed
        params={"per_page": count},                    # changed
        timeout=10,
    )
    r.raise_for_status()
    return r.json()


def read_repos(path: Path) -> Iterator[Repo]:
    data = json.loads(path.read_text(encoding="utf-8"))
    for item in data:
        yield Repo(item["name"], item["stargazers_count"])   # changed


def main() -> None:
    CACHE_DIR.mkdir(exist_ok=True)
    cache_file = CACHE_DIR / "repos_cache.json"

    if not cache_file.exists():
        print("No cache. Fetching...")
        cache_file.write_text(
            json.dumps(fetch_repos(5), indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
    else:
        print(f"Reading cache: {cache_file}")

    for repo in read_repos(cache_file):
        print("  " + repo.line())


main()

---

That is the toolkit. Go back to **[python_toolkit.html](python_toolkit.html)** for the
diagrams and the 中文 version, or keep going with `python_apis_async.ipynb` for
`async` / `await`.